In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from google.colab import drive
drive.mount('/content/drive')


# File name: Death_rates_for_suicide__by_sex__race__Hispanic_origin__and_age__United_States.csv
original_df = pd.read_csv('/content/drive/MyDrive/Death_rates_for_suicide__by_sex__race__Hispanic_origin__and_age__United_States.csv')

df = original_df.copy()

# Explore the data

#df.describe()
#df.info()
#df.dtypes
#print(df)
#df.isnull().sum().sort_values(ascending=False)

# Clean the data

# Dropped the Flag Column
df = df.drop(['FLAG', 'INDICATOR',], axis=1)

new_column_names = {
    'STUB_NAME' : 'demographic_term', 'UNIT_NUM' : 'unit_num', 'STUB_NAME_NUM' : 'demo_id', 'STUB_LABEL' : 'demo_type',
       'STUB_LABEL_NUM' : 'demo_type_id', 'AGE_NUM' : 'age_id',  'ESTIMATE' : 'suicide_rate' }

df = df.rename(columns=new_column_names)

#df.info()

# Keep only the crude rate and drop the age-adjusted duplicate.
# The raw file reports two UNIT variants under the same demo_type_id/YEAR
# (crude and age-adjusted), which is why the duplicate check below was
# firing: the regression/interpolation was silently blending two
# different rate definitions into one series.
# Crude is the one to standardize on. Only 25 of 161 demo_type_id groups
# even report an age-adjusted rate (it's only calculated at aggregate
# levels: totals, sex, race, never for a specific age bracket, since
# adjusting for age doesn't apply once you're already stratified by
# age). Filtering to age-adjusted would silently drop every age-bracket
# row, so crude is the one that keeps all the data.
print("UNIT value counts before filtering:")
print(df['UNIT'].value_counts())

crude_mask = df['UNIT'].str.contains('crude', case=False, na=False)
print(f"\nKeeping {crude_mask.sum():,} of {len(df):,} rows tagged crude "
      f"(dropping {(~crude_mask).sum():,} age-adjusted rows).")
df = df[crude_mask].drop(columns=['UNIT', 'unit_num']).reset_index(drop=True)

# Drop the 2018-only "Single race" comparison rows (demo_id 8-11).
# NCHS re-tabulated 2018 under the newer single-race race/Hispanic-origin
# categories as a one-time side-by-side comparison against the standard
# bridged-race categories (demo_id 0-7) that make up the full 1950-2018
# series. Both sets reuse the same demo_type_id codes for a given
# demographic, so every 2018 row in demo_id 4-7 has a duplicate
# "Single race" counterpart under demo_id 8-11. That's the other half of
# the (demo_type_id, YEAR) collisions the check below still caught after
# the crude/age-adjusted filter. The bridged-race categories are what
# the rest of the series and this notebook's race/Hispanic mapping are
# built on, so keep those and drop the one-off rows.
single_race_mask = df['demo_id'].isin([8, 9, 10, 11])
print(f"Dropping {single_race_mask.sum():,} 'Single race' 2018-only comparison rows "
      f"(demo_id 8-11); they duplicate demo_type_id codes already covered by the "
      f"bridged-race series (demo_id 0-7).")
df = df[~single_race_mask].reset_index(drop=True)

# Define your functions

# Validate: exactly one row per (demo_type_id, YEAR) before we start
# imputing, since the regression/interpolation below groups by
# demo_type_id and treats YEAR as the series index. Two rows sharing a
# (demo_type_id, YEAR) pair would get silently blended together.
# Any pair still duplicated at this point isn't a classification
# artifact (those were filtered out above); it's a genuine duplicate
# publication in the raw NCHS file (same demo_type_id/AGE/YEAR, two
# different reported ESTIMATE values). There's no principled way to
# pick a "correct" row in that case, so we average suicide_rate within
# each duplicated pair. It's an auditable tie-break that keeps every
# group in the series instead of silently keeping whichever row
# happened to come first.
dupe_mask = df.duplicated(subset=['demo_type_id', 'YEAR'], keep=False)
if dupe_mask.any():
    print(f"WARNING: {dupe_mask.sum()} rows share a (demo_type_id, YEAR) pair, "
          "averaging suicide_rate within each pair. Raw rows:")
    print(df.loc[dupe_mask, ['demographic_term', 'demo_type', 'demo_type_id', 'YEAR', 'suicide_rate']]
            .sort_values(['demo_type_id', 'YEAR']))
    agg_map = {c: 'first' for c in df.columns if c not in ('demo_type_id', 'YEAR', 'suicide_rate')}
    agg_map['suicide_rate'] = 'mean'
    df = df.groupby(['demo_type_id', 'YEAR'], as_index=False).agg(agg_map)
    assert not df.duplicated(subset=['demo_type_id', 'YEAR'], keep=False).any(), \
        "Still have duplicate (demo_type_id, YEAR) pairs after averaging"
    print("OK: every (demo_type_id, YEAR) pair is unique after averaging duplicates.")
else:
    print("OK: every (demo_type_id, YEAR) pair is unique.")

MIN_POINTS_FOR_TREND = 4

def impute_with_regression(group):
    """
    Fills missing suicide_rate values in a group, sorted by YEAR:
      - interior gaps (bounded by real values on both sides) are filled
        by linear interpolation over YEAR, since suicide rates aren't
        linear across decades and interpolation only reasons about the
        nearest known points instead of the whole series
      - leading/trailing gaps need extrapolation, so they fall back to
        a per-group linear regression on YEAR, but only when there are
        at least MIN_POINTS_FOR_TREND observed points to fit a trend on;
        otherwise they're left null and handled by the dropna() below
    Predictions are clamped at 0 (a rate can't be negative) and every
    filled row is flagged in 'suicide_rate_imputed' so it stays
    distinguishable from an actually-observed value downstream.
    """
    group = group.sort_values('YEAR').copy()
    group['suicide_rate_imputed'] = False

    if not group['suicide_rate'].isnull().any():
        return group

    was_missing = group['suicide_rate'].isnull()

    # Interior gaps: interpolate over YEAR
    group['suicide_rate'] = group.set_index('YEAR')['suicide_rate'] \
        .interpolate(method='index', limit_area='inside').values

    # Whatever's still missing is a leading/trailing gap. Extrapolate if we can.
    predict_mask = group['suicide_rate'].isnull()
    train_data = group.dropna(subset=['suicide_rate'])

    if predict_mask.any() and len(train_data) >= MIN_POINTS_FOR_TREND:
        X_train = train_data[['YEAR']].values
        y_train = train_data['suicide_rate'].values
        X_predict = group.loc[predict_mask, ['YEAR']].values

        model = LinearRegression()
        model.fit(X_train, y_train)
        predictions = np.clip(model.predict(X_predict), a_min=0, a_max=None)
        group.loc[predict_mask, 'suicide_rate'] = predictions

    group.loc[was_missing & group['suicide_rate'].notnull(), 'suicide_rate_imputed'] = True
    return group

# Apply the function to each group and store the results in a new DataFrame
n_missing_before = df['suicide_rate'].isnull().sum()
imputed_df = df.groupby('demo_type_id').apply(impute_with_regression, include_groups=True)

# Reset the index to clean up the DataFrame
imputed_df = imputed_df.reset_index(drop=True)
n_imputed = imputed_df['suicide_rate_imputed'].sum()
n_missing_after = imputed_df['suicide_rate'].isnull().sum()
print(f"suicide_rate missing before: {n_missing_before:,}")
print(f"suicide_rate filled by interpolation/regression: {n_imputed:,}")
print(f"suicide_rate still missing (not enough points to trust a trend): {n_missing_after:,}")

# Drop the Leftover Nulls (Nulls are less than 2% of the Data)
cleaned_idf = imputed_df.copy()
cleaned_idf = cleaned_idf.dropna(subset=['suicide_rate'])


#cleaned_idf['Race'] = None

# 1. Create a "Dictionary Table" for ID-based mappings
# This is where you can easily add your Age Groups later

# 1. Mapping for specific IDs

import re

# 1. Standardize the text to be clean of whitespace
cleaned_idf['demo_type'] = cleaned_idf['demo_type'].astype(str).str.strip()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
UNIT value counts before filtering:
UNIT
Deaths per 100,000 resident population, crude           5578
Deaths per 100,000 resident population, age-adjusted     812
Name: count, dtype: int64

Keeping 5,578 of 6,390 rows tagged crude (dropping 812 age-adjusted rows).
Dropping 96 'Single race' 2018-only comparison rows (demo_id 8-11); they duplicate demo_type_id codes already covered by the bridged-race series (demo_id 0-7).
       demographic_term                                          demo_type  demo_type_id  YEAR  suicide_rate
2645  Sex, age and race       Male: Black or African American: 45-64 years         5.124  2018          11.3
2646  Sex, age and race       Male: Black or African American: 45-64 years         5.124  2018          18.9
2688  Sex, age and race  Male: Black or African American: 65 years and ...         5.125  2018           8.7
2689  Sex,

/tmp/ipykernel_1568/3782974194.py:148: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imputed_df = df.groupby('demo_type_id').apply(impute_with_regression, include_groups=True)


In [36]:
!pip install -q pymysql cryptography
print('done')

done


In [37]:
print("FLAG value counts (including NaN):")
print(original_df['FLAG'].value_counts(dropna=False))
print()
print(f"Rows with a FLAG set: {original_df['FLAG'].notna().sum():,} / {len(original_df):,}")
print()
print("Does a FLAG correlate with a missing ESTIMATE (suicide_rate)?")
print(original_df.assign(_missing_estimate=original_df['ESTIMATE'].isnull()).groupby('FLAG', dropna=False)['_missing_estimate'].mean())
print()
print("Sample of rows sharing a (demo_type_id, YEAR) pair:")
print(df[df.duplicated(subset=['demo_type_id', 'YEAR'], keep=False)].sort_values(['demo_type_id', 'YEAR']).head(20))


FLAG value counts (including NaN):
FLAG
NaN    5484
...     645
*       261
Name: count, dtype: int64

Rows with a FLAG set: 906 / 6,390

Does a FLAG correlate with a missing ESTIMATE (suicide_rate)?
FLAG
*      1.0
...    1.0
NaN    0.0
Name: _missing_estimate, dtype: float64

Sample of rows sharing a (demo_type_id, YEAR) pair:
Empty DataFrame
Columns: [demo_type_id, YEAR, demographic_term, demo_id, demo_type, YEAR_NUM, AGE, age_id, suicide_rate]
Index: []


In [38]:
if 'UNIT' not in df.columns:
    print("Skipping: UNIT was already dropped by the crude-rate filter in the "
          "pipeline cell above; this diagnostic predates that fix and is now stale.")
else:
    print("UNIT value counts:")
    print(df['UNIT'].value_counts())
    print()

    # For each demo_type_id, which UNIT variants are present?
    unit_presence = df.groupby('demo_type_id')['UNIT'].agg(lambda s: sorted(s.unique()))
    both = unit_presence.apply(lambda x: len(x) == 2)
    only_one = unit_presence.apply(lambda x: len(x) == 1)

    print(f"demo_type_id groups with BOTH unit variants: {both.sum()}")
    print(f"demo_type_id groups with ONLY ONE unit variant: {only_one.sum()}")
    print()
    print("demo_type_id groups with only one UNIT variant (id -> [unit(s)]):")
    print(unit_presence[only_one])
    print()

    # Specifically check the age-bracket ids from id_to_group
    age_ids = [1.1, 1.2, 1.21, 1.22, 1.3, 1.31, 1.32, 1.4, 1.41, 1.42, 1.5, 1.51, 1.52, 1.53]
    print("UNIT variants present for age-bracket demo_type_ids:")
    print(unit_presence[unit_presence.index.isin(age_ids)])

Skipping: UNIT was already dropped by the crude-rate filter in the pipeline cell above; this diagnostic predates that fix and is now stale.


In [39]:
if 'UNIT' not in df.columns:
    print("Skipping: UNIT was already dropped by the crude-rate filter in the "
          "pipeline cell above; this diagnostic predates that fix and is now stale.")
else:
    print("UNIT value counts:")
    print(df['UNIT'].value_counts())
    print()

    # For each demo_type_id, which UNIT variants are present?
    unit_presence = df.groupby('demo_type_id')['UNIT'].agg(lambda s: sorted(s.unique()))
    both = unit_presence.apply(lambda x: len(x) == 2)
    only_one = unit_presence.apply(lambda x: len(x) == 1)

    print(f"demo_type_id groups with BOTH unit variants: {both.sum()}")
    print(f"demo_type_id groups with ONLY ONE unit variant: {only_one.sum()}")
    print()
    print("demo_type_id groups with only one UNIT variant (id -> [unit(s)]):")
    print(unit_presence[only_one])
    print()

    # Specifically check the age-bracket ids from id_to_group
    age_ids = [1.1, 1.2, 1.21, 1.22, 1.3, 1.31, 1.32, 1.4, 1.41, 1.42, 1.5, 1.51, 1.52, 1.53]
    print("UNIT variants present for age-bracket demo_type_ids:")
    print(unit_presence[unit_presence.index.isin(age_ids)])

Skipping: UNIT was already dropped by the crude-rate filter in the pipeline cell above; this diagnostic predates that fix and is now stale.


In [40]:
import os
import ssl
import numpy as np
from sqlalchemy import create_engine, text as sql_text
from dotenv import load_dotenv

load_dotenv()

# --- Safety net: clamp any negative imputed rates to 0 ---
cleaned_idf['suicide_rate'] = cleaned_idf['suicide_rate'].clip(lower=0)

# Assuming final_df was meant to be cleaned_idf after imputation
fixed_df = cleaned_idf.copy()
fixed_df['suicide_rate'] = fixed_df['suicide_rate'].clip(lower=0)
fixed_df = fixed_df.rename(columns={'demo_type': 'demographic_group'})

# --- Validate before export ---
# known_groups = set(id_to_group.values()) | set(race_definitions.values())
assert (fixed_df['suicide_rate'] >= 0).all(), "Negative suicide_rate found in fixed_df"
# assert set(fixed_df['demographic_group'].dropna().unique()) <= known_groups, "Unexpected group value in fixed_df"
print(f"fixed_df: {len(fixed_df):,} rows across {fixed_df['demographic_group'].nunique()} groups: "
      f"{sorted(fixed_df['demographic_group'].dropna().unique())}")

# --- Secure DB connection ---
USER = os.getenv('DB_USER')
PASSWORD = os.getenv('DB_PASS')
HOST = os.getenv('DB_HOST')
PORT = os.getenv('DB_PORT', '20511')
DATABASE = os.getenv('DB_NAME')
SSL_CA = os.getenv('DB_SSL_CA')  # path to Aiven's CA cert, if you've downloaded one

missing = [n for n, v in [('DB_USER', USER), ('DB_PASS', PASSWORD), ('DB_HOST', HOST), ('DB_NAME', DATABASE)] if not v]
if missing:
    raise RuntimeError(f"Missing required environment variables: {missing}. Set them in your .env file.")

if SSL_CA:
    ssl_context = ssl.create_default_context(cafile=SSL_CA)
else:
    print("WARNING: DB_SSL_CA not set - falling back to disabled certificate verification. "
          "Download the CA cert from your Aiven console and set DB_SSL_CA to close this gap.")
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(connection_string, connect_args={'ssl': ssl_context})

# --- Export the corrected data ---
try:
    fixed_df.to_sql(name='suicide_data', con=engine, if_exists='replace', index=False, chunksize=1000)
    with engine.begin() as conn:
        conn.execute(sql_text("CREATE INDEX idx_suicide_data_year ON suicide_data (YEAR)"))
        conn.execute(sql_text("CREATE INDEX idx_suicide_data_group ON suicide_data (demographic_group(191))"))
    print("Successfully exported corrected data and created indexes!")
except Exception as e:
    print(f"Error: {e}")

fixed_df: 5,375 rows across 138 groups: ['10-14 years', '15-19 years', '15-24 years', '20-24 years', '25-34 years', '25-44 years', '35-44 years', '45-54 years', '45-64 years', '55-64 years', '65 years and over', '65-74 years', '75-84 years', '85 years and over', 'All persons', 'Female', 'Female: 10-14 years', 'Female: 15-19 years', 'Female: 15-24 years', 'Female: 20-24 years', 'Female: 25-34 years', 'Female: 25-44 years', 'Female: 35-44 years', 'Female: 45-54 years', 'Female: 45-64 years', 'Female: 55-64 years', 'Female: 65 years and over', 'Female: 65-74 years', 'Female: 75-84 years', 'Female: 85 years and over', 'Female: American Indian or Alaska Native', 'Female: American Indian or Alaska Native: 15-24 years', 'Female: American Indian or Alaska Native: 25-44 years', 'Female: American Indian or Alaska Native: 45-64 years', 'Female: Asian or Pacific Islander', 'Female: Asian or Pacific Islander: 15-24 years', 'Female: Asian or Pacific Islander: 25-44 years', 'Female: Asian or Pacific 

In [41]:
import os
print(os.listdir('/content'))
if 'env' in os.listdir('/content') and '.env' not in os.listdir('/content'): os.rename('/content/env', '/content/.env')
from dotenv import load_dotenv
load_dotenv(override=True)
ca = os.getenv('DB_SSL_CA')
print('DB_SSL_CA set:', bool(ca), '| points to existing file:', os.path.isfile(ca) if ca else False)

['.config', '.ipynb_checkpoints', '.env', 'drive', 'sample_data']
DB_SSL_CA set: False | points to existing file: False


In [42]:
import os
if 'env' in os.listdir('/content') and '.env' not in os.listdir('/content'): os.rename('/content/env', '/content/.env')
print(os.listdir('/content'))

['.config', '.ipynb_checkpoints', '.env', 'drive', 'sample_data']


In [43]:
import socket
try: print('google.com ->', socket.gethostbyname('google.com'))
except Exception as e: print('google.com FAILED:', e)
try: print('aiven host ->', socket.gethostbyname('mysql-3e311278-jthomas5040-c1a5.l.aivencloud.com'))
except Exception as e: print('aiven host FAILED:', e)

google.com -> 74.125.130.101
aiven host -> 165.232.181.147


In [44]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

dupe_mask2 = df.duplicated(subset=['demo_type_id', 'YEAR'], keep=False)
dupes = df[dupe_mask2].sort_values(['demo_type_id', 'YEAR'])
print(f"Duplicate rows (crude-only data): {len(dupes)}")
print(f"Distinct demo_type_id values involved: {dupes['demo_type_id'].nunique()}")
print()
print("Duplicate count per demo_type_id:")
print(dupes.groupby('demo_type_id').size())
print()
print("Sample rows (first 20):")
print(dupes.head(20))


Duplicate rows (crude-only data): 0
Distinct demo_type_id values involved: 0

Duplicate count per demo_type_id:
Series([], dtype: int64)

Sample rows (first 20):
Empty DataFrame
Columns: [demo_type_id, YEAR, demographic_term, demo_id, demo_type, YEAR_NUM, AGE, age_id, suicide_rate]
Index: []


In [45]:
def norm_age(raw):
    return raw.replace(' years and over','+').replace(' years','')

def norm_race(raw):
    m = {'Black or African American':'Black','Asian or Pacific Islander':'Asian','American Indian or Alaska Native':'Native American or Alaska Native'}
    return m.get(raw, raw)

rows = []
src = imputed_df.rename(columns={'suicide_rate_imputed':'Imputed'})

for _, r in src[src['demographic_term']=='Total'].iterrows():
    rows.append({'YEAR':int(r['YEAR']),'Rate':r['suicide_rate'],'Imputed':bool(r['Imputed']),'Breakdown':'Total','Sex':'All','AgeGroup':'All','Race':'All'})

for _, r in src[src['demographic_term']=='Age'].iterrows():
    rows.append({'YEAR':int(r['YEAR']),'Rate':r['suicide_rate'],'Imputed':bool(r['Imputed']),'Breakdown':'Age','Sex':'All','AgeGroup':norm_age(r['demo_type']),'Race':'All'})

for _, r in src[src['demographic_term']=='Sex'].iterrows():
    rows.append({'YEAR':int(r['YEAR']),'Rate':r['suicide_rate'],'Imputed':bool(r['Imputed']),'Breakdown':'Sex','Sex':r['demo_type'],'AgeGroup':'All','Race':'All'})

for _, r in src[src['demographic_term']=='Sex and age'].iterrows():
    sex, age = r['demo_type'].split(': ',1)
    rows.append({'YEAR':int(r['YEAR']),'Rate':r['suicide_rate'],'Imputed':bool(r['Imputed']),'Breakdown':'Sex and Age','Sex':sex,'AgeGroup':norm_age(age),'Race':'All'})

for _, r in src[src['demographic_term']=='Sex and race'].iterrows():
    sex, race = r['demo_type'].split(': ',1)
    rows.append({'YEAR':int(r['YEAR']),'Rate':r['suicide_rate'],'Imputed':bool(r['Imputed']),'Breakdown':'Sex and Race/Ethnicity','Sex':sex,'AgeGroup':'All','Race':norm_race(race)})

hisp = src[(src['demographic_term']=='Sex and race and Hispanic origin') & (src['demo_type'].str.contains('Hispanic or Latino: All races'))]
for _, r in hisp.iterrows():
    sex = r['demo_type'].split(':')[0]
    rows.append({'YEAR':int(r['YEAR']),'Rate':r['suicide_rate'],'Imputed':bool(r['Imputed']),'Breakdown':'Sex and Race/Ethnicity','Sex':sex,'AgeGroup':'All','Race':'Hispanic or Latino'})

other_df = pd.DataFrame(rows)
print(other_df.shape)
print(other_df['Breakdown'].value_counts())
print(other_df.isna().sum())
print(other_df.head(3).to_string())

(2310, 7)
Breakdown
Sex and Age               1176
Age                        588
Sex and Race/Ethnicity     420
Sex                         84
Total                       42
Name: count, dtype: int64
YEAR         0
Rate         0
Imputed      0
Breakdown    0
Sex          0
AgeGroup     0
Race         0
dtype: int64
   YEAR  Rate  Imputed Breakdown  Sex AgeGroup Race
0  1950  11.4    False     Total  All      All  All
1  1960  10.6    False     Total  All      All  All
2  1970  11.6    False     Total  All      All  All


In [46]:
other_df.to_csv('/content/drive/MyDrive/suicide_rates_richdata.csv', index=False)
print('saved rich csv', other_df.shape)

saved rich csv (2310, 7)


In [47]:
overall_1950 = fixed_df[fixed_df['YEAR']==1950]['suicide_rate'].mean()
overall_2018 = fixed_df[fixed_df['YEAR']==2018]['suicide_rate'].mean()
print(f"All-persons-level avg rate, 1950: {overall_1950:.2f} per 100k -> 2018: {overall_2018:.2f} per 100k")
group_avg = fixed_df.groupby('demographic_group')['suicide_rate'].mean().sort_values(ascending=False)
print(group_avg.round(2).to_string())
male_trend = fixed_df[fixed_df['demographic_group']=='Male'].sort_values('YEAR')[['YEAR','suicide_rate']]
female_trend = fixed_df[fixed_df['demographic_group']=='Female'].sort_values('YEAR')[['YEAR','suicide_rate']]
print('Male 1950 vs 2018:', male_trend.iloc[0].suicide_rate, '->', male_trend.iloc[-1].suicide_rate)
print('Female 1950 vs 2018:', female_trend.iloc[0].suicide_rate, '->', female_trend.iloc[-1].suicide_rate)

All-persons-level avg rate, 1950: 15.01 per 100k -> 2018: 15.70 per 100k
demographic_group
Male: White: 85 years and over                                                       58.63
Male: 85 years and over                                                              54.58
Male: White: 75-84 years                                                             45.60
Male: Not Hispanic or Latino: American Indian or Alaska Native: 15-24 years          43.84
Male: 75-84 years                                                                    42.55
Male: Not Hispanic or Latino: American Indian or Alaska Native: 25-44 years          42.11
Male: Not Hispanic or Latino: White: 65 years and over                               39.48
Male: White: 65 years and over                                                       37.62
Male: American Indian or Alaska Native: 15-24 years                                  35.49
Male: 65 years and over                                                              35.01

In [48]:
print(sorted(df[df['demographic_term']=='Sex and race']['demo_type'].unique().tolist()))
print(df['demographic_term'].unique().tolist())

['Female: American Indian or Alaska Native', 'Female: Asian or Pacific Islander', 'Female: Black or African American', 'Female: White', 'Male: American Indian or Alaska Native', 'Male: Asian or Pacific Islander', 'Male: Black or African American', 'Male: White']
['Total', 'Age', 'Sex', 'Sex and age', 'Sex and race', 'Sex, age and race', 'Sex and race and Hispanic origin', 'Sex, age and race and Hispanic origin']


In [49]:
fixed_df.round(2).to_json('/content/drive/MyDrive/suicide_data_export.json', orient='records')
print('saved to drive')

saved to drive


In [50]:
import json
records = fixed_df.round(2).to_dict(orient='records')
print(json.dumps(records))

[{"demo_type_id": 0.0, "YEAR": 1950, "demographic_term": "Total", "demo_id": 0, "demographic_group": "All persons", "YEAR_NUM": 1, "AGE": "All ages", "age_id": 0.0, "suicide_rate": 11.4, "suicide_rate_imputed": false}, {"demo_type_id": 0.0, "YEAR": 1960, "demographic_term": "Total", "demo_id": 0, "demographic_group": "All persons", "YEAR_NUM": 2, "AGE": "All ages", "age_id": 0.0, "suicide_rate": 10.6, "suicide_rate_imputed": false}, {"demo_type_id": 0.0, "YEAR": 1970, "demographic_term": "Total", "demo_id": 0, "demographic_group": "All persons", "YEAR_NUM": 3, "AGE": "All ages", "age_id": 0.0, "suicide_rate": 11.6, "suicide_rate_imputed": false}, {"demo_type_id": 0.0, "YEAR": 1980, "demographic_term": "Total", "demo_id": 0, "demographic_group": "All persons", "YEAR_NUM": 4, "AGE": "All ages", "age_id": 0.0, "suicide_rate": 11.9, "suicide_rate_imputed": false}, {"demo_type_id": 0.0, "YEAR": 1981, "demographic_term": "Total", "demo_id": 0, "demographic_group": "All persons", "YEAR_NUM": 

In [51]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
dupe_mask4 = df.duplicated(subset=['demo_type_id', 'YEAR'], keep=False)
dupes3 = df[dupe_mask4].sort_values(['demo_type_id', 'YEAR'])
print(f"Remaining duplicate rows: {len(dupes3)}")
print(dupes3)
print(fixed_df.columns.tolist()); print(fixed_df.head(3).to_string())

Remaining duplicate rows: 0
Empty DataFrame
Columns: [demo_type_id, YEAR, demographic_term, demo_id, demo_type, YEAR_NUM, AGE, age_id, suicide_rate]
Index: []
['demo_type_id', 'YEAR', 'demographic_term', 'demo_id', 'demographic_group', 'YEAR_NUM', 'AGE', 'age_id', 'suicide_rate', 'suicide_rate_imputed']
   demo_type_id  YEAR demographic_term  demo_id demographic_group  YEAR_NUM       AGE  age_id  suicide_rate  suicide_rate_imputed
0           0.0  1950            Total        0       All persons         1  All ages     0.0          11.4                 False
1           0.0  1960            Total        0       All persons         2  All ages     0.0          10.6                 False
2           0.0  1970            Total        0       All persons         3  All ages     0.0          11.6                 False
